# 🌋 TRM and DRM on Réunion Island

Our Locality: Piton de la Fournaise, Réunion Island

[Piton de la Fournaise](https://en.wikipedia.org/wiki/Piton_de_la_Fournaise) is one of the world's most active volcanoes, located on Réunion Island in the western Indian Ocean at **21.1°S, 55.5°E**. It erupts frequently, producing basaltic lava flows that cool in the ambient geomagnetic field and acquire a thermal remanent magnetization (TRM).

Imagine that sediments also accumulate in nearby lakes, recording the same field through depositional remanent magnetization (DRM). Let's think about TRM and DRM from the same locality.

In [ ]:
!pip install pmagpy
import numpy as np
import matplotlib.pyplot as plt
from pmagpy import ipmag
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider
import folium
import logging
logging.getLogger("matplotlib.axes._base").setLevel(logging.ERROR)

## Interactive map fun

Let's have a look at the volcano!

In [ ]:
# Interactive satellite map of our study site
m = folium.Map(location=[-21.244, 55.708], zoom_start=10, tiles=None,
               control_scale=True)

folium.TileLayer(
    tiles="http://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google",
    name="Google Satellite",
    overlay=False,
).add_to(m)

# Mark Piton de la Fournaise
folium.Marker(
    [-21.244, 55.708],
    popup='<b>Piton de la Fournaise</b><br>21.2°S, 55.7°E<br>One of the most active volcanoes on Earth',
    tooltip='Piton de la Fournaise',
    icon=folium.Icon(color='red', icon='fire', prefix='fa')
).add_to(m)

m

## Local geomagnetic field at Réunion

Let's start by calculating what the geomagnetic field looks like at Réunion right now, using the **International Geomagnetic Reference Field (IGRF)** model.

### 🧭 A reminder about directions: Declination and Inclination

A paleomagnetic direction is described by two angles:

- **Declination ($D$)**: the azimuthal angle measured clockwise from geographic north, ranging from 0° to 360°. A declination of 0° points north, 90° points east, 180° points south, and 270° points west.

- **Inclination ($I$)**: the angle from the horizontal plane. Positive inclination points *downward* into the Earth (as in the Northern Hemisphere); negative inclination points *upward* out of the Earth (as in the Southern Hemisphere). Inclination ranges from +90° (straight down) to −90° (straight up).

Together, $D$ and $I$ fully specify a unit vector direction in 3D space. You can think of declination as the compass direction of the horizontal component, and inclination as the dip angle below (or above) the horizon.

Under the **geocentric axial dipole (GAD) hypothesis** — the foundational assumption of paleomagnetism — the time-averaged geomagnetic field corresponds to a dipole aligned with Earth's spin axis. The GAD model predicts:

$$\tan(I) = 2\tan(\lambda)$$

where $\lambda$ is the geographic latitude. This means inclination systematically steepens from the equator (where $I = 0°$) to the poles (where $|I| = 90°$). In the Southern Hemisphere, inclinations are negative.

In [ ]:
# Piton de la Fournaise, Réunion Island
site_lat = -21.115
site_lon = 55.532
date = 2025.0

# Calculate the IGRF field direction and intensity (the present-day field)
igrf_result = ipmag.igrf([date, 0, site_lat, site_lon])
field_dec = igrf_result[0]
field_inc = igrf_result[1]
field_int = igrf_result[2]

# Calculate the expected GAD (geocentric axial dipole) field direction
# Under GAD: declination = 0° (aligned with geographic north) and
# inclination is set by latitude through the dipole formula
gad_dec = 0.0
gad_inc = ipmag.inc_from_lat(site_lat)

In [ ]:
print(f"IGRF Field at Réunion ({site_lat}°N, {site_lon}°E) in {date:.0f}:")
print(f"  Declination: {field_dec:.1f}°")
print(f"  Inclination: {field_inc:.1f}°")
print(f"  Intensity:   {field_int:.0f} nT")
print(f"\nGeocentric Axial Dipole expected field at latitude {site_lat}°:")
print(f"  Declination: {gad_dec:.1f}°")
print(f"  Inclination: {gad_inc:.1f}°")

## Plotting Directions: The Equal-Area Stereonet

Paleomagnetic directions live on a sphere, so we need a projection to plot them on a flat screen. The standard projection in paleomagnetism is the **equal-area (Schmidt) projection**, also called a stereonet.

The conventions are:
- **North** is at the top, **East** is to the right
- The **center** of the circle represents straight down ($I = +90°$)
- The **rim** of the circle represents the horizontal plane ($I = 0°$)
- **Solid symbols** = lower hemisphere (positive inclination, pointing downward)
- **Open symbols** = upper hemisphere (negative inclination, pointing upward)

Since our locality is in the Southern Hemisphere, the field direction points *upward* (negative inclination), so we expect to see open symbols.

Let's plot the expected field direction at Réunion:

In [ ]:
ipmag.plot_net(fignum=1)

ipmag.plot_di(dec=[field_dec], inc=[field_inc],
              color='#D55E00', marker='*', markersize=200,
              label='IGRF (present-day)')

ipmag.plot_di(dec=[gad_dec], inc=[gad_inc],
              color='#009E73', marker='*', markersize=200,
              label='GAD (time-averaged)')

plt.title(f'Geomagnetic field at Réunion\n'
          f'IGRF: Dec={field_dec:.1f}°, Inc={field_inc:.1f}° | '
          f'GAD: Dec={gad_dec:.1f}°, Inc={gad_inc:.1f}°', fontsize=11)
plt.legend(loc='lower center')
plt.show()

## 🌡️ Thermoremanent Magnetization (TRM)

As you now know, a lava flow cools through the **Curie temperature** of its magnetic minerals (typically magnetite, ~580°C), the magnetic moments of individual grains become locked in and become aligned with the ambient geomagnetic field at the time of cooling. This produces a **thermoremanent magnetization (TRM)** that is parallel to the field.

As you also now know, the geomagnetic field is not static as it undergoes **secular variation**, with the non-dipole components changing on timescales of hundreds to thousands of years. Each lava flow erupted at a different time records a slightly different instantaneous field direction. We saw above that today's IGRF field at Réunion differs from the GAD prediction by over 16° in inclination.

Over sufficiently long time intervals (typically $>10^5$ years), secular variation averages out and the mean direction converges on the GAD prediction. We can model this scatter of instantaneous directions around the GAD field using a **Fisher distribution** — the spherical analog of a Gaussian distribution on a line. *This is an oversimplification, but ok for the moment.*

The Fisher distribution is characterized by the **precision parameter** $\kappa$ (kappa):
- Higher $\kappa$ → more tightly clustered directions
- Lower $\kappa$ → more dispersed directions
- $\kappa = 0$ → uniformly random directions on the sphere

Let's simulate 30 lava flows erupted at Réunion over a period long enough to sample secular variation. Each flow faithfully records the instantaneous field, and the scatter around the GAD direction represents secular variation:

In [ ]:
# Simulation parameters
n_flows = 30       # number of lava flows sampling secular variation
kappa_sv = 25      # precision parameter (secular variation scatter)

# Generate Fisher-distributed TRM directions around the GAD field direction
# Each direction represents a lava flow recording the instantaneous field
# at a different point in time — the scatter is secular variation
trm_dirs = ipmag.fishrot(k=kappa_sv, n=n_flows, dec=gad_dec, inc=gad_inc)

# Plot on a stereonet
ipmag.plot_net(fignum=2)
ipmag.plot_di(di_block=trm_dirs, color='#0072B2', markersize=30, label='TRM flows')
ipmag.plot_di(dec=[gad_dec], inc=[gad_inc],
              color='darkred', marker='*', markersize=200, label='GAD field')
plt.title(f'Simulated TRM directions (n={n_flows}, κ={kappa_sv})', fontsize=12)
plt.show()

## Fisher Statistics

When we have a set of paleomagnetic directions, we need to calculate their mean and quantify the scatter. On a sphere, we use **Fisher statistics** (Fisher, 1953).

The key quantities are:

- **Mean direction** ($D$, $I$): the direction of the *resultant vector* — the vector sum of all unit vectors. This is the best estimate of the true direction.

- **Resultant length** ($R$): the magnitude of the vector sum. If all $n$ directions are identical, $R = n$; if they are randomly scattered, $R \approx 0$.

- **Precision parameter** ($\kappa$): estimated as $\hat{\kappa} = (n-1)/(n-R)$. Higher $\kappa$ means tighter clustering.

- **95% confidence cone** ($\alpha_{95}$): the half-angle of a cone around the mean direction within which we are 95% confident the *true mean* lies. Smaller $\alpha_{95}$ = more precise estimate.

$\alpha_{95}$ depends on both $\kappa$ and $n$ — more directions and tighter clustering both shrink the confidence cone.

Let's calculate Fisher statistics for our simulated TRM data:

In [ ]:
# Calculate Fisher statistics for the TRM directions
trm_mean = ipmag.fisher_mean(di_block=trm_dirs)

print("Fisher statistics for TRM directions:")
ipmag.print_direction_mean(trm_mean)

print(f"\nGAD field direction: Dec={gad_dec:.1f}°, Inc={gad_inc:.1f}°")

# Plot with mean and alpha95 confidence cone
ipmag.plot_net(fignum=3)
ipmag.plot_di(di_block=trm_dirs, color='#0072B2', markersize=30,
              label = 'lava flow directions')
ipmag.plot_di_mean(trm_mean['dec'], trm_mean['inc'], trm_mean['alpha95'],
                   color='blue', marker='D', markersize=50,
                   label = 'mean of lava flow directions')
ipmag.plot_di(dec=[gad_dec], inc=[gad_inc],
              color='darkred', marker='*', markersize=200,
              label='expected direction')
plt.legend(loc='lower center')
plt.show()

## Depositional Remanent Magnetization (DRM) and Inclination Shallowing

Sediments acquire magnetization through a very different process. As grains settle and accumulate, they tend to align with the ambient field producing a **depositional remanent magnetization (DRM)**.

However, DRM has a systematic bias: **inclination shallowing**. Elongate or platy magnetic grains tend to lie flat on the bedding plane as they settle and compact. This rotates the recorded direction toward the horizontal, making the measured inclination shallower than the true field inclination.

The relationship is described by the **flattening formula** (King, 1955):

$$\tan(I_{\text{DRM}}) = f \cdot \tan(I_{\text{field}})$$

where:
- $I_{\text{field}}$ is the true field inclination
- $I_{\text{DRM}}$ is the recorded (shallowed) inclination
- $f$ is the **flattening factor**: $f = 1$ means perfect recording (no shallowing), $f < 1$ means inclination is shallowed

Typical values of $f$ in natural sediments range from about 0.4 to 0.8, depending on lithology.

Let's apply inclination shallowing to our simulated directions and compare TRM with DRM:

In [ ]:
# Apply moderate flattening using ipmag.squish
f_value = 0.6
drm_incs = ipmag.squish(list(trm_dirs[:, 1]), f_value)
drm_dirs = np.column_stack([trm_dirs[:, 0], drm_incs])

# Calculate Fisher stats for DRM
drm_mean = ipmag.fisher_mean(di_block=drm_dirs)

# Side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.5))

# TRM (left panel)
ipmag.plot_net(ax=ax1)
plt.sca(ax1)
ipmag.plot_di(di_block=trm_dirs, color='#0072B2', markersize=30)
ipmag.plot_di_mean(trm_mean['dec'], trm_mean['inc'], trm_mean['alpha95'],
                   color='#0072B2', marker='D', markersize=50)
ipmag.plot_di(dec=[gad_dec], inc=[gad_inc],
              color='darkred', marker='*', markersize=200)
ax1.set_title(f"TRM (faithful recording)\n"
              f"Mean Inc={trm_mean['inc']:.1f}°, α95={trm_mean['alpha95']:.1f}°",
              fontsize=11)

# DRM (right panel)
ipmag.plot_net(ax=ax2)
plt.sca(ax2)
ipmag.plot_di(di_block=drm_dirs, color='#E69F00', markersize=30)
ipmag.plot_di_mean(drm_mean['dec'], drm_mean['inc'], drm_mean['alpha95'],
                   color='#E69F00', marker='D', markersize=50)
ipmag.plot_di(dec=[gad_dec], inc=[gad_inc],
              color='darkred', marker='*', markersize=200)
ax2.set_title(f"DRM (f={f_value})\n"
              f"Mean Inc={drm_mean['inc']:.1f}°, α95={drm_mean['alpha95']:.1f}°",
              fontsize=11)

plt.tight_layout()
plt.show()

print(f"Inclination comparison:")
print(f"  GAD field:   {gad_inc:.1f}°")
print(f"  TRM mean:    {trm_mean['inc']:.1f}°")
print(f"  DRM mean:    {drm_mean['inc']:.1f}° (shallowed by {abs(trm_mean['inc']) - abs(drm_mean['inc']):.1f}°)")

### Paleolatitude comparison: TRM vs. DRM

Often the goal of measuring paleomagnetic inclinations is to determine **paleolatitude** — where the site was when the rocks formed. We can invert the dipole formula to get:

$$\lambda = \arctan\left(\frac{\tan(I)}{2}\right)$$

The $\alpha_{95}$ confidence cone on the mean direction propagates into an asymmetric uncertainty on the paleolatitude estimate: the upper and lower bounds on inclination ($I \pm \alpha_{95}$) map to different-sized latitude intervals because of the nonlinear dipole equation.

Let's compute paleolatitudes from both the TRM and DRM mean inclinations using `ipmag.lat_from_inc` and compare them to the actual site latitude:

In [ ]:
# Paleolatitude from TRM (with uncertainty from alpha95)
plat_trm, plat_trm_max, plat_trm_min = ipmag.lat_from_inc(
    trm_mean['inc'], a95=trm_mean['alpha95'])

# Paleolatitude from DRM (with uncertainty from alpha95)
plat_drm, plat_drm_max, plat_drm_min = ipmag.lat_from_inc(
    drm_mean['inc'], a95=drm_mean['alpha95'])

print(f"Actual site latitude: {site_lat:.1f}°")
print(f"GAD expected paleolatitude: {ipmag.lat_from_inc(gad_inc):.1f}°")
print()
print(f"TRM paleolatitude: {plat_trm:.1f}° ({plat_trm_min:.1f}° to {plat_trm_max:.1f}°)")
print(f"DRM paleolatitude: {plat_drm:.1f}° ({plat_drm_min:.1f}° to {plat_drm_max:.1f}°)")
print()
print(f"The TRM estimate places the site at {abs(plat_trm):.1f}°S — "
      f"within {abs(abs(plat_trm) - abs(site_lat)):.1f}° of the true latitude.")
print(f"The DRM estimate places the site at {abs(plat_drm):.1f}°S — "
      f"{abs(abs(site_lat) - abs(plat_drm)):.1f}° too close to the equator.")

## Interactive Exploration: Squishing the Inclination

Use the slider below to explore how the flattening factor $f$ affects DRM directions. Watch how the directions migrate toward the rim of the stereonet (shallower inclination) as $f$ decreases:

- **$f = 1.0$**: DRM perfectly matches TRM (no flattening)
- **$f = 0.6$**: moderate flattening (typical of many sediments)
- **$f = 0.4$**: severe flattening (heavily compacted sediments)

In [ ]:
def plot_drm_interactive(f=0.6):
    """Plot DRM directions on an equal-area net with adjustable inclination flattening.

    Applies an inclination flattening factor (f) to the original directional
    dataset (trm_dirs) using ipmag.squish, computes Fisher statistics for the
    flattened population, and displays the resulting directions and mean on
    a stereonet. The unflattened (f = 1.0) mean direction is shown as a
    faint reference marker for comparison.

    Parameters
    ----------
    f : float
        Inclination flattening factor (1.0 = no flattening, < 1 = shallowing).
    """
    # Apply flattening to get DRM directions
    drm_incs = ipmag.squish(list(trm_dirs[:, 1]), f)
    drm = np.column_stack([trm_dirs[:, 0], drm_incs])

    drm_stats = ipmag.fisher_mean(di_block=drm)

    plat_drm = ipmag.lat_from_inc(drm_stats['inc'])
    plat_trm = ipmag.lat_from_inc(trm_mean['inc'])

    # Side-by-side comparison
    fig, (ax1) = plt.subplots(1, 1, figsize=(5.5, 5.5))

    ipmag.plot_net(ax=ax1)
    plt.sca(ax1)
    ipmag.plot_di(di_block=drm, color='#E69F00', markersize=30)
    ipmag.plot_di_mean(drm_stats['dec'], drm_stats['inc'], drm_stats['alpha95'],
                       color='#E69F00', marker='D', markersize=100)
    ipmag.plot_di(dec=[trm_mean['dec']], inc=[trm_mean['inc']],
                  color='#0072B2', marker='D', markersize=80, alpha=0.3)
    ax1.set_title(f"DRM (f={f:.2f})\n"
                  f"Mean Inc={drm_stats['inc']:.1f}°\n"
                  f"Paleolatitude: {plat_drm:.1f}°",
                  fontsize=11)

    plt.tight_layout()
    plt.show()

# Create the interactive slider
f_slider = FloatSlider(
    value=1.0, min=0.4, max=1.0, step=0.05,
    description='Flattening f:',
    layout=widgets.Layout(width='400px')
)

interact(plot_drm_interactive, f=f_slider)